# 02 - Limpieza

Prepara el dataset limpio que usan los notebooks 03 a 06.

In [1]:
from pathlib import Path
import warnings

from matplotlib.colors import LinearSegmentedColormap

warnings.filterwarnings("ignore", category=PendingDeprecationWarning, module="seaborn")

SPOTIFY_GREEN = "#1DB954"
SPOTIFY_BLACK = "#191414"
SPOTIFY_GRAY = "#535353"
SPOTIFY_PALETTE = ["#1DB954", "#1ED760", "#0E7A37", "#73E68C", "#117A37", "#0A5C29"]
SPOTIFY_SEQ = LinearSegmentedColormap.from_list("spotify_seq", ["#FFFFFF", "#1DB954", "#0A4A22"])
SPOTIFY_DIVERGE = LinearSegmentedColormap.from_list("spotify_div", ["#191414", "#FFFFFF", "#1DB954"])


def apply_spotify_theme():
    import matplotlib as mpl
    import seaborn as sns

    sns.set_theme(style="whitegrid", palette=SPOTIFY_PALETTE)
    mpl.rcParams.update({
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "axes.titleweight": "bold",
        "axes.titlecolor": SPOTIFY_BLACK,
        "axes.labelcolor": SPOTIFY_BLACK,
        "axes.edgecolor": "#C9C9C9",
        "grid.color": "#ECECEC",
        "text.color": SPOTIFY_BLACK,
        "font.size": 11,
        "axes.titlesize": 14,
        "figure.titlesize": 16,
        "figure.titleweight": "bold",
    })


PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_RAW = PROJECT_ROOT / "data" / "raw" / "dataset.csv"
DATA_CLEAN = PROJECT_ROOT / "data" / "processed" / "spotify_tracks_clean.csv"

RANDOM_STATE = 42

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv(DATA_RAW)
print(df.shape)
display(df.head())

(114000, 21)


,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,...,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,...,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,...,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,...,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,...,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


## Duplicados y columnas sobrantes

`Unnamed: 0` es el índice del CSV y se elimina. También se quitan los `track_id` duplicados.

In [3]:
df_clean = df.copy()
if "Unnamed: 0" in df_clean.columns:
    df_clean = df_clean.drop(columns=["Unnamed: 0"])

before_rows = len(df_clean)
df_clean = df_clean.drop_duplicates(subset=["track_id"]).reset_index(drop=True)
print(f"Filas antes: {before_rows:,}")
print(f"Filas después de quitar duplicados: {len(df_clean):,}")

Filas antes: 114,000
Filas después de quitar duplicados: 89,741


## Imputación

Numéricas con mediana, categóricas con moda.

In [4]:
numeric_cols = df_clean.select_dtypes(include=np.number).columns.tolist()
categorical_cols = df_clean.select_dtypes(exclude=np.number).columns.tolist()

for col in numeric_cols:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

for col in categorical_cols:
    mode = df_clean[col].mode(dropna=True)
    fill_value = mode.iloc[0] if not mode.empty else "Unknown"
    df_clean[col] = df_clean[col].fillna(fill_value)

print(f"Faltantes restantes: {df_clean.isna().sum().sum()}")

Faltantes restantes: 0


## Nuevas columnas

- `duration_min`: duración en minutos
- `is_explicit`: explicit como 0/1
- `popularity_class`: terciles Low / Medium / High (para clasificación)

In [5]:
df_clean["duration_min"] = df_clean["duration_ms"] / 60000
df_clean["is_explicit"] = df_clean["explicit"].astype(int)

labels = ["Low", "Medium", "High"]
df_clean["popularity_class"] = pd.qcut(df_clean["popularity"], q=3, labels=labels)

display(df_clean[["popularity", "duration_min", "is_explicit", "popularity_class"]].head())
display(df_clean["popularity_class"].value_counts().to_frame("count"))

,popularity,duration_min,is_explicit,popularity_class
0,73,3.844433,0,High
1,55,2.493500,0,High
2,57,3.513767,0,High
3,71,3.365550,0,High
4,82,3.314217,0,High


,count
popularity_class,
Low,31659
High,29870
Medium,28212


## Vista previa del escalado

El escalado definitivo se hace en cada notebook después del train/test split,
para no usar información del test.

In [6]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

audio_features = [
    "danceability", "energy", "loudness", "speechiness", "acousticness",
    "instrumentalness", "liveness", "valence", "tempo", "duration_min",
    "key", "mode", "time_signature", "is_explicit"
]

standard_preview = pd.DataFrame(
    StandardScaler().fit_transform(df_clean[audio_features]),
    columns=audio_features
).describe().loc[["mean", "std"]]

normalized_preview = pd.DataFrame(
    MinMaxScaler().fit_transform(df_clean[audio_features]),
    columns=audio_features
).describe().loc[["min", "max"]]

display(standard_preview.round(3))
display(normalized_preview.round(3))

,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_min,key,mode,time_signature,is_explicit
mean,-0.0,0.0,-0.0,0.0,0.0,-0.0,-0.0,0.0,-0.0,-0.0,-0.0,0.0,0.0,-0.0
std,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0


,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_min,key,mode,time_signature,is_explicit
min,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
max,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0


## One-hot de `track_genre`

In [7]:
genre_encoded_preview = pd.get_dummies(df_clean[["track_genre"]], drop_first=True)
print(f"Columnas dummy de género: {genre_encoded_preview.shape[1]:,}")
display(genre_encoded_preview.head())

Columnas dummy de género: 112


,track_genre_afrobeat,track_genre_alt-rock,track_genre_alternative,track_genre_ambient,track_genre_anime,track_genre_black-metal,track_genre_bluegrass,track_genre_blues,track_genre_brazil,track_genre_breakbeat,...,track_genre_spanish,track_genre_study,track_genre_swedish,track_genre_synth-pop,track_genre_tango,track_genre_techno,track_genre_trance,track_genre_trip-hop,track_genre_turkish,track_genre_world-music
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [8]:
DATA_CLEAN.parent.mkdir(parents=True, exist_ok=True)
df_clean.to_csv(DATA_CLEAN, index=False)
print(f"Guardado en: {DATA_CLEAN}")
print(f"Shape final: {df_clean.shape}")

Guardado en: /Users/skulltula/Documents/proyecto-ia-tc3002b/data/processed/spotify_tracks_clean.csv
Shape final: (89741, 23)
